# Broker Auth: Zerodha (Kite Connect)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivikasavnish/algo-trading-notebooks/blob/main/notebooks/02_broker_auth_zerodha_kite.ipynb)

Authenticate and place a demo order via Kite Connect over your static IP.

Part 02 of 36 in the [ServLoci algo/options trading notebook series](https://comm.servloci.in/docs) — full index in `notebooks/README.md`.

## Setup

In [ ]:
# Get your dedicated static IPv6 + SOCKS5 credentials free:
#   https://comm.servloci.in/register        (or /auth/google?free=1 for an instant trial)
# Your api_key / api_secret pair shows up in the portal after signup:
#   https://comm.servloci.in/user
!pip install -q "requests[socks]"
!curl -sL https://comm.servloci.in/sdk/servloci.py -o servloci.py

import os
from servloci import ServLoci

SERVLOCI_API_KEY = os.environ.get("SERVLOCI_API_KEY", "dhan:1000000001")   # broker:client_id
SERVLOCI_API_SECRET = os.environ.get("SERVLOCI_API_SECRET", "")            # from the portal — leave blank to run this notebook in demo mode

sl = None
if SERVLOCI_API_SECRET:
    sl = ServLoci(api_key=SERVLOCI_API_KEY, api_secret=SERVLOCI_API_SECRET)
    print("ServLoci configured:", sl.host, sl.port)
else:
    print("SERVLOCI_API_SECRET not set — running in demo mode (no live proxy calls).")

**Support level:** Self-service — Kite Connect apps are approved instantly from the developer console; add your ServLoci IPv6 there.

Indian broker APIs authenticate trading sessions with an OAuth-style
two-step handshake, not a single long-lived key. You redirect the user (in
practice, yourself) to a broker login page, the broker redirects back with a
short-lived `request_token`, and you exchange that token — together with your
app's `api_secret` — for an `access_token` that's valid for actual order
calls. The `api_key`/`api_secret` pair identifies *your app*; the
`access_token` identifies *an authenticated session* and, at every broker
covered here, expires at end of trading day regardless of activity. That
forced daily re-login is a deliberate session-security control, common across
SEBI-regulated broker APIs, so a stolen access token has a shelf life measured
in hours, not indefinitely.

Kite Connect is the most self-service of the four: app creation, API key
issuance, and IP allowlisting all happen immediately in the developer console,
with no manual review step. That makes it a common first choice for testing
an integration, though the API itself carries a separate subscription fee
independent of brokerage.

In [ ]:
if sl:
    sl.attach()  # BEFORE importing kiteconnect — it creates a requests.Session at import time
    from kiteconnect import KiteConnect

    kite = KiteConnect(api_key=os.environ.get("KITE_API_KEY", ""))
    print("Login URL:", kite.login_url())
    # After the browser redirect, exchange request_token for an access_token:
    # data = kite.generate_session(request_token, api_secret=os.environ["KITE_API_SECRET"])
    # kite.set_access_token(data["access_token"])
    # order_id = kite.place_order(tradingsymbol="INFY", exchange=kite.EXCHANGE_NSE,
    #                              transaction_type=kite.TRANSACTION_TYPE_BUY, quantity=1,
    #                              order_type=kite.ORDER_TYPE_MARKET, product=kite.PRODUCT_MIS,
    #                              variety=kite.VARIETY_REGULAR)
else:
    print("Demo mode — set SERVLOCI_API_SECRET above to run this against your Zerodha account.")

Install the broker SDK separately: `!pip install -q kiteconnect`. Full whitelist steps: [https://comm.servloci.in/docs/brokers](https://comm.servloci.in/docs).

---

« Previous: [ServLoci SDK Quickstart](01_servloci_sdk_quickstart.ipynb)  
Next: [Broker Auth: Dhan](03_broker_auth_dhan.ipynb) »

Try the concepts above interactively: [Options Strategy Builder](https://comm.servloci.in/tools/strategy-builder) · [Docs](https://comm.servloci.in/docs) · [Get your static IP](https://comm.servloci.in/register)